# Mathematical Foundations of Linear Regression

**Training a linear regression model is equivalent to solving the following optimization problem:**

Given training samples $\{x^{(i)}, y^{(i)}\}_{i=1}^m$, finding coefficients is done by solving the following optimization problem

$$ \frac{1}{m} \sum_{i=1}^m (w_0 + w_1x^{(i)}_1 + ... + w_dx^{(i)}_d - y^{(i)})^2. $$

In this lecture, we will introduce several methods to solve the optimization problem. To keep Math simple, we consider the following minimization problem:

$$\mathop{\mathbf{minimize}}_{k,b} \frac{1}{m}\sum_{i=1}^{m} (kx^{(i)} + b - y_i)^2, $$
where m is the number of data samples.

Our implementation can be generalized to high-dimensional case. We leave tne generalization to the assignment.

# Linear Algebra:

Using Linear Algebra, we could write the optimization problem as
$$\mathop{\mathbf{minimize}}_{c\in\mathbb{R}^2} \| \mathbf{y} - \mathbf{X}c\|_2^2, $$
where 
$$ \mathbf{y} = \begin{bmatrix} y_1 \\ \vdots y_m \end{bmatrix}, \mbox{ and }  \mathbf{X} = \begin{bmatrix} 1 & x^{(1)} \\ \vdots & \vdots \\ 1 & x^{(m)} \end{bmatrix}.$$
Moreover, our coefficient vector $c$ contains $[b,k]$.

This problem is called a least-squares problem, and it has a nice formula:
$$c = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$$

**`sklearn.LinearRegression` uses normal equation.**

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import time

In [4]:
m = 50
X = np.random.normal(0,1,size=(m,1))
y = 2*X-3+np.random.normal(0,0.1,size=(m,1))

# normal equation
X_aug = np.column_stack((np.ones((m,1)),X))
c1 = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
print(c1)

# Sklearn
reg = LinearRegression().fit(X,y)
print(reg.coef_, reg.intercept_)

[[-3.00239768]
 [ 1.98840333]]
[[1.98840333]] [-3.00239768]


**Challenges:**
1. Inverting a matrix $A$ is computationally inefficient. Suppose that $A\in\mathbb{R}^{m\times m}$, the time complexity is $\mathcal{O}(m^3)$.

2. Training most machine learning models does not result in solving a least-squares problem, and hence we cannot use it.

# GD and SGD

The linear regression model is favorable because it is easy to interprete and solve. Unfortunately, not all models are easy to solve. Usually, we face "hard" optimization problems when we train machine learning models in the sense that there is no nice explicit formula as training linear regression models.

We will introcude gradient descent (GD) and stochastic gradient descent (SGD) here, which are methods for solving optimization problem.

### GD

Mathematically, optimization problem takes the following form

$$\mathop{\mathbf{minimize}/\mathbf{maximize}} f(x) $$
where $f(x)$ is called **objective function**.

GD is an iterative algorithm which produces a sequence of points ${x_1,x_2,...,x_n}$ closing to our target $x^*$. The updating rule is

$$ x_{t+1} = x_{t} - \eta \nabla f(x_t),$$
where $\eta$ is called stepsize or learning rate. Here, $x_t$ is a vector. If $x_t$ is a real number, then the gradient $\nabla f(x_t)$ reduces to derivative $f'(x_t)$ ( I assume that you know how to compute derivative and gradient for a given function). Gradient tells you the direction that the objective function decreases fastest. Stepsize determines how much you move each time.  

#### Example of GD

In [ ]:
def f(x):
    "x is a real number"
    return x**2

def grad_f(x):
    "x is a real number"
    return 2*x

def GD(x, eta, fun):
    """
    x is a real number
    eta: stepsize, a real number
    fun: is a function, which is the derivative of your objective function
    """ 
    return x - eta*fun(x)

In [ ]:
x = 5         # initial point or starting point: x_0
eta = 0.05    # stepsize
iteration = 200 # number of iterations

for i in range(iteration):
    x = GD(x,eta,grad_f)
    if i%20 == 0:
        print(f'After iteration {i+1}, the updated point is x={x}')

## GD for linear regression

Let's consider the linear regression problem again, and we want to use GD to train the linear regression model.

In [ ]:
import time

In [ ]:
m = int(1e5)
x = np.linspace(-10,10,m)
y = 2*x-3+np.random.randn(m)    # the last term represents random noise

#initial values 
k,b = 1, -2

# step size or learning rate
eta = 0.01

# number of iterations
n_iter = 400

# GD

start = time.time()
for t in range(n_iter+1):
    
    # compute the gradient
    grad_k = sum( 2*(k*x+b-y) * x )/m
    grad_b = sum( 2*(k*x+b-y) ) /m
    
    # updating rule
    k = k - eta*grad_k
    b = b - eta*grad_b
    
    # compute the loss
    L = sum( (k*x+b-y)**2 ) / m
    
    # print outputs
    if t%50 ==0:
        print(f"Step {t}".ljust(10), end=" ")
        print(f"L =  {L:.2f}".ljust(12), end=" ")
        print(f"k = {k:.3f}".ljust(15), end=" ")
        print(f"b = {b:.3f}".ljust(15), end=" ")
        print(" ")
        
stop = time.time()
print(f"Computational time is {stop-start} seconds.")

## Claim: GD is slow.

Notice that this is a simple machine learning example in the sense that 1) number of training data is small, 2) model is simple because there are only 2 parameters, and 3) data is 1D.

## Reason:

The gradient sums over $m$ terms. It is computationally expensive to compute gradient when $m$ is large (number of observations).

## Solution: stochastic gradient descent

Instead of using $m$ data points, we select $b$ data points and do gradient descent using selected data points ($b < m$). 

Those data points are randomly selected, this is where the word 'stochastic' comes from. 

Terminology:
- batch size: how many data points are selected to run GD ($b$ above).

Some remarks:
1. In machine learning task, gradient is a summation over $m$ terms.
2. Number of observations $m$ is huge (big data)
3. Many ways to do SGD, we only introduce the most common one.



A nice explanation on SGD with batch size 1: https://nbviewer.org/github/PhilChodrow/PIC16B/blob/master/lectures/math/optimization.ipynb

In [ ]:
m = int(1e5)
x = np.linspace(-10,10,m)
y = 2*x-3+np.random.randn(m)    # the last term represents random noise

#initial values 
k,b = 1, -2

# step size or learning rate
eta = 0.01

# number of iterations
n_iter = 400

# SGD

start = time.time()
for t in range(n_iter+1):
    
    # select data samples to approximate the gradient
    j = np.random.choice(m,1)[0]
    x_j, y_j = x[j], y[j]
    
    # compute the gradient
    grad_k = 2*(k*x_j+b-y_j) * x_j
    grad_b = 2*(k*x_j+b-y_j)
    
    # updating rule
    k = k - eta*grad_k
    b = b - eta*grad_b
    
    # compute the loss
    L = sum( (k*x+b-y)**2 ) / m
    
    # print outputs
    if t%50 ==0:
        print(f"Step {t}".ljust(10), end=" ")
        print(f"L =  {L:.2f}".ljust(12), end=" ")
        print(f"k = {k:.3f}".ljust(15), end=" ")
        print(f"b = {b:.3f}".ljust(15), end=" ")
        print(" ")
        
stop = time.time()
print(f"Computational time is {stop-start} seconds.")

## SGD regression in Python 

In [ ]:
X, Y = make_regression(n_samples=20000, n_features = 5, noise=1)

## Using sklearn command - LinearRegression
reg = LinearRegression().fit(X, Y)
print(reg.intercept_, reg.coef_)

## Using sklearn command - SGDRegressor
from sklearn.linear_model import SGDRegressor

reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None).fit(X,Y)
print(reg.intercept_, reg.coef_)

# Why do we discuss SGD?

Stochastic gradient descent (SGD) and its variants are widely used in training machine learning models. When we discuss neural networks, we will borrow some terminologies from SGD. This is the main reason that I touch SGD here.